# Tutorial 4: Molecule (Drug) Embeddings

This notebook covers generating embeddings for **small molecules / drugs**
with `embpy`. Molecules are represented as **SMILES** strings.

Available molecule embedding models:

| Model key | Architecture | Notes |
|---|---|---|
| `chemberta2MTR` | ChemBERTa-2 (MTR) | Transformer, multi-task regression head |
| `chemberta2MLM` | ChemBERTa-2 (MLM) | Transformer, masked-language-model head |
| `molformer_base` | MolFormer | Large-scale molecular transformer |

Additionally, `embpy` supports **RDKit fingerprints** as a non-learned baseline.

If you have a drug *name* instead of a SMILES, use the `DrugResolver`
(see [Tutorial 1](01_identifiers_and_preprocessing.ipynb)) to convert it first,
or let `embed_molecule` handle it automatically.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Resolve drug names to SMILES

Most embedding models expect canonical SMILES. The `DrugResolver` maps
common drug names to SMILES via PubChem.

In [ ]:
from embpy.resources.drug_resolver import DrugResolver

drug_resolver = DrugResolver()

drugs = {
    "aspirin":    drug_resolver.name_to_smiles("aspirin"),
    "ibuprofen":  drug_resolver.name_to_smiles("ibuprofen"),
    "caffeine":   drug_resolver.name_to_smiles("caffeine"),
    "paclitaxel": drug_resolver.name_to_smiles("paclitaxel"),
}

for name, smi in drugs.items():
    print(f"  {name:12s} → {smi}")

## 2. Embed a single molecule

In [ ]:
MODEL = "chemberta2MTR"

aspirin_smi = drugs["aspirin"]
print(f"Aspirin SMILES: {aspirin_smi}")

emb = embedder.embed_molecule(
    identifier=aspirin_smi,
    model=MODEL,
    pooling_strategy="mean",
)

print(f"Embedding shape: {emb.shape}")
print(f"Embedding dtype: {emb.dtype}")

## 3. Batch molecule embedding

In [ ]:
smiles_list = [smi for smi in drugs.values() if smi is not None]
names_list = [n for n, s in drugs.items() if s is not None]

embeddings = embedder.embed_molecules_batch(
    identifiers=smiles_list,
    model=MODEL,
    pooling_strategy="mean",
)

for name, emb in zip(names_list, embeddings):
    if emb is not None:
        print(f"  {name:12s} → dim {emb.shape[0]}")
    else:
        print(f"  {name:12s} → FAILED")

## 4. Comparing molecule models

Different models produce embeddings in different spaces and dimensions.

In [ ]:
import numpy as np

test_smi = drugs["caffeine"]

for m in ["chemberta2MTR", "chemberta2MLM", "molformer_base"]:
    try:
        e = embedder.embed_molecule(test_smi, model=m, pooling_strategy="mean")
        print(f"  {m:20s} → dim {e.shape[0]:,}, L2 = {np.linalg.norm(e):.2f}")
    except Exception as exc:
        print(f"  {m:20s} → {exc}")

## 5. Similarity between molecules

Embedding-based cosine similarity can capture structural and functional
relatedness between drugs.

In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

# Embed two structurally related NSAIDs (aspirin & ibuprofen)
# and one very different molecule (caffeine)
embs = {}
for name in ["aspirin", "ibuprofen", "caffeine"]:
    smi = drugs[name]
    if smi:
        embs[name] = embedder.embed_molecule(smi, model=MODEL, pooling_strategy="mean")

if len(embs) == 3:
    print(f"aspirin  vs ibuprofen: {cosine_sim(embs['aspirin'], embs['ibuprofen']):.4f}")
    print(f"aspirin  vs caffeine:  {cosine_sim(embs['aspirin'], embs['caffeine']):.4f}")
    print(f"ibuprofen vs caffeine: {cosine_sim(embs['ibuprofen'], embs['caffeine']):.4f}")

## 6. Reverse lookup: SMILES → drug name

In [ ]:
# What is this molecule?
unknown_smi = "CC(=O)Oc1ccccc1C(O)=O"

primary = drug_resolver.smiles_to_primary_name(unknown_smi)
all_names = drug_resolver.smiles_to_names(unknown_smi, top_k=5)

print(f"Primary name: {primary}")
print(f"All names:    {all_names}")

## 7. Building an embedding matrix from an AnnData

If you have a perturbation-screen `AnnData` whose `.obs` contains a
column with drug names or SMILES, use
`PerturbationProcessor.build_molecule_embedding_matrix` to embed them
all at once. It resolves names → SMILES automatically, embeds, and
stores the result in `.obsm`.

In [ ]:
import anndata as ad
import pandas as pd
from embpy.pp.basic import PerturbationProcessor

pp = PerturbationProcessor(embedder=embedder)

# Simulate a screen AnnData with drug metadata
screen_obs = pd.DataFrame({
    "treatment":  ["aspirin", "ibuprofen", "caffeine", "paclitaxel"],
    "dose_uM":    [10.0, 5.0, 1.0, 0.1],
    "cell_line":  ["A549", "A549", "HeLa", "HeLa"],
})
adata_screen = ad.AnnData(obs=screen_obs)

# Embed – just point at the .obs column with drug names
adata_emb = pp.build_molecule_embedding_matrix(
    adata=adata_screen,
    column="treatment",        # which .obs column has drug identifiers
    id_type="drug_name",       # they are common names, not SMILES
    model="chemberta2MTR",
    obsm_key="X_chemberta",
)

print("Result AnnData (original metadata preserved):")
print(adata_emb.obs)
print(f"\nEmbedding matrix: {adata_emb.obsm['X_chemberta'].shape}")
print(f"Successful: {adata_emb.obs['embedded'].sum()} / {len(adata_emb)}")

## Summary

| What | How |
|---|---|
| Drug name → SMILES | `DrugResolver().name_to_smiles("aspirin")` |
| Single molecule embedding | `embed_molecule(smiles, model)` |
| Batch molecule embedding | `embed_molecules_batch(smiles_list, model)` |
| Embed from AnnData column | `pp.build_molecule_embedding_matrix(adata=..., column="treatment")` |
| Embed from a list | `pp.build_molecule_embedding_matrix(identifiers=[...])` |
| SMILES → drug name | `DrugResolver().smiles_to_names(smiles)` |

Next: [Tutorial 5 – Text Embeddings](05_text_embeddings.ipynb)